# Google Geocoding CSV ツール (Colab)\n\nこのノートブックは `geocode_csv.py` を Google Colab 上で実行するためのものです。

## 1) API キーを設定

In [ ]:
import os
import getpass

os.environ['GOOGLE_MAPS_API_KEY'] = getpass.getpass('Google Maps API Key: ')
print('API キーを環境変数 GOOGLE_MAPS_API_KEY に設定しました。')

## 2) 入力CSVをアップロード

In [ ]:
from google.colab import files

uploaded = files.upload()
print('アップロード済みファイル:', list(uploaded.keys()))

## 3) ジオコーディング処理をノートブック内で定義
`.py` を別ファイル生成せず、このノートブックだけで実行できます。


In [ ]:
import csv
import json
import time
import urllib.error
import urllib.parse
import urllib.request
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Tuple

GEOCODE_URL = "https://maps.googleapis.com/maps/api/geocode/json"

@dataclass
class GeocodeResult:
    lat: float
    lng: float
    formatted_address: str

@dataclass
class GeocodeFailure:
    status: str
    message: str

@dataclass
class GeocodeResponse:
    result: Optional[GeocodeResult]
    failure: Optional[GeocodeFailure] = None

class GoogleGeocoder:
    def __init__(self, api_key: str, sleep_seconds: float = 0.1, timeout_seconds: int = 10) -> None:
        self.api_key = api_key
        self.sleep_seconds = sleep_seconds
        self.timeout_seconds = timeout_seconds
        self._cache: Dict[str, GeocodeResponse] = {}

    def geocode(self, address: str) -> GeocodeResponse:
        address = address.strip()
        if not address:
            return GeocodeResponse(result=None, failure=GeocodeFailure(status="EMPTY_ADDRESS", message="住所が空です"))

        if address in self._cache:
            return self._cache[address]

        params = urllib.parse.urlencode(
            {"address": address, "key": self.api_key, "language": "ja", "region": "jp"}
        )
        req = urllib.request.Request(url=f"{GEOCODE_URL}?{params}", method="GET")

        try:
            with urllib.request.urlopen(req, timeout=self.timeout_seconds) as resp:
                payload = json.loads(resp.read().decode("utf-8"))
        except urllib.error.URLError as exc:
            raise RuntimeError(f"APIリクエスト失敗: address='{address}', error={exc}") from exc

        status = payload.get("status")
        if status == "OK":
            top = payload["results"][0]
            loc = top["geometry"]["location"]
            response = GeocodeResponse(
                result=GeocodeResult(
                    lat=loc["lat"],
                    lng=loc["lng"],
                    formatted_address=top.get("formatted_address", ""),
                )
            )
            self._cache[address] = response
            time.sleep(self.sleep_seconds)
            return response

        error_message = payload.get("error_message") or "Geocoding APIが結果を返しませんでした"
        response = GeocodeResponse(
            result=None,
            failure=GeocodeFailure(status=status or "UNKNOWN", message=error_message),
        )
        self._cache[address] = response
        time.sleep(self.sleep_seconds)
        return response


def _read_rows(path: str) -> Tuple[List[str], List[dict]]:
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames is None:
            raise ValueError("CSVヘッダーが見つかりません")
        return list(reader.fieldnames), list(reader)


def _ensure_columns(headers: List[str], extra_cols: Iterable[str]) -> List[str]:
    merged = list(headers)
    for col in extra_cols:
        if col not in merged:
            merged.append(col)
    return merged


def process_csv(
    input_csv: str,
    output_csv: str,
    geocoder: GoogleGeocoder,
    address_col: str = "住所",
    lat_col: str = "緯度",
    lng_col: str = "経度",
    formatted_address_col: str = "正規化住所",
    failure_reason_col: str = "ジオコーディング失敗理由",
):
    headers, rows = _read_rows(input_csv)
    out_headers = _ensure_columns(headers, [lat_col, lng_col, formatted_address_col, failure_reason_col])

    failures = []
    success = 0

    for i, row in enumerate(rows, start=2):
        address = (row.get(address_col) or "").strip()
        response = geocoder.geocode(address)

        if response.result is None:
            status = response.failure.status if response.failure else "UNKNOWN"
            message = response.failure.message if response.failure else "原因不明"
            reason = f"{status}: {message}"
            row[lat_col] = ""
            row[lng_col] = ""
            row[formatted_address_col] = ""
            row[failure_reason_col] = reason
            failures.append({"row": i, "address": address, "reason": reason})
        else:
            row[lat_col] = str(response.result.lat)
            row[lng_col] = str(response.result.lng)
            row[formatted_address_col] = response.result.formatted_address
            row[failure_reason_col] = ""
            success += 1

    with open(output_csv, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=out_headers)
        writer.writeheader()
        writer.writerows(rows)

    return {
        "total": len(rows),
        "success": success,
        "failed": len(failures),
        "failures": failures,
        "output_csv": output_csv,
    }



## 4) 変換を実行（失敗理由をその場で確認）


In [ ]:
# 必要に応じてファイル名を変更してください
INPUT_CSV = 'サンプル_飲食店_豊中_緯度経度なし.csv'
OUTPUT_CSV = 'output_with_latlng.csv'


In [ ]:
API_KEY = os.environ.get('GOOGLE_MAPS_API_KEY', '').strip()
if not API_KEY:
    raise ValueError('GOOGLE_MAPS_API_KEY が未設定です。最初のセルを実行してください。')

geocoder = GoogleGeocoder(api_key=API_KEY, sleep_seconds=0.1, timeout_seconds=10)
summary = process_csv(INPUT_CSV, OUTPUT_CSV, geocoder)

print(
    f"完了: total={summary['total']}, success={summary['success']}, "
    f"failed={summary['failed']}, output='{summary['output_csv']}'"
)

if summary['failures']:
    try:
        import pandas as pd
        display(pd.DataFrame(summary['failures']))
    except Exception:
        for failure in summary['failures']:
            print(f"WARN row={failure['row']} address='{failure['address']}' -> {failure['reason']}")
else:
    print('失敗行はありませんでした。')



## 5) 出力CSVをダウンロード

In [ ]:
files.download(OUTPUT_CSV)
